# 02 — Investigador: verificación de cobertura

**Objetivo:** entender cómo el agente Investigador decide si el tema pedido está
cubierto en el documento, y qué hace cuando no lo está.

**Qué vas a ver:**
1. Los **dos protocolos inyectados**: `Recuperador` (búsqueda determinística en
   Chroma) y `GeneradorEstructurado` (redacción de la aclaración).
2. El caso con cobertura: si hay un chunk sobre el umbral, confirma sin llamar al LLM.
3. El caso sin cobertura: redacta un mensaje de aclaración anclado al documento.
4. Los errores del contrato: tema vacío, chunk inválido, salida inválida.
5. (Opcional) la aclaración con un LLM real, si tienes proveedor.

**Qué vas a aprender:** por qué la verificación de cobertura NO pasa por el LLM
(es determinística) y por qué el modelo solo se usa para redactar la aclaración.


In [ ]:
import sys
from pathlib import Path

# El notebook corre desde notebooks/; el código vive en la raíz del repo.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))
# `src/` tambien entra al path porque algunos módulos importan de forma "pelada"
# (gotcha del AGENTS.md). Es un puente para el notebook, no el estilo.
sys.path.insert(0, str(RAIZ / "src"))

print("Raiz del repo:", RAIZ)


## 1. Los dos protocolos inyectados

`investigar(...)` recibe un `recuperador` (busca chunks con su similitud) y un
`generador` (redacta). El agente no sabe de dónde vienen: por eso se prueba con
dobles, sin red ni claves.


In [ ]:
from typing import Any

from pydantic import BaseModel

from src.agentes.agente_investigador import (
    UMBRAL_COBERTURA,
    ChunkConScore,
    investigar,
)


class FakeGenerator:
    """Doble que cumple `GeneradorEstructurado` y no toca ningún proveedor."""

    def __init__(self, salida: Any):
        self.salida = salida
        self.prompt = ""
        self.output_model: type[BaseModel] | None = None

    async def generate(self, *, prompt: str, output_model: type[BaseModel]) -> Any:
        self.prompt = prompt
        self.output_model = output_model
        if isinstance(self.salida, Exception):
            raise self.salida
        return self.salida


class FakeRecuperador:
    """Doble de `rag/vectorstore.py`: devuelve los chunks que le pasemos."""

    def __init__(self, chunks: list[Any]):
        self.chunks = chunks
        self.consulta: str | None = None
        self.top_k: int | None = None

    def buscar(self, consulta: str, top_k: int = 5) -> list[Any]:
        self.consulta = consulta
        self.top_k = top_k
        return self.chunks


## 2. Con cobertura: confirma sin llamar al LLM

Un chunk con `score >= UMBRAL_COBERTURA` (0.78) basta para confirmar. Fíjate que
el generador queda intacto: **no se usó**, porque no hay nada que redactar.


In [ ]:
TITULO = "Manual de redes en la nube"
CHUNK_MATCH = {"id": "chunk-1", "texto": "Una VCN es una red privada.", "score": 0.91}
CHUNK_BAJO = {"id": "chunk-2", "texto": "Texto poco relacionado.", "score": 0.40}

recuperador = FakeRecuperador([CHUNK_MATCH, CHUNK_BAJO])
generador = FakeGenerator({"mensaje": "no debería usarse"})

resultado = await investigar("VCN", recuperador, generador, TITULO)
print("fuente_confirmada:", resultado.fuente_confirmada)
print("chunks confirmados:", [c.id for c in resultado.chunks_fuente_confirmados])
print("¿se llamó al LLM?:", generador.output_model is not None)


## 3. El umbral: justo en 0.78 también confirma

El corte es `>=`, no `>`. Un chunk exactamente en `UMBRAL_COBERTURA` cuenta.


In [ ]:
en_umbral = {"id": "c", "texto": "justo en el umbral", "score": UMBRAL_COBERTURA}
resultado = await investigar(
    "VCN", FakeRecuperador([en_umbral]), FakeGenerator({}), TITULO
)
print("umbral:", UMBRAL_COBERTURA, "| confirmada:", resultado.fuente_confirmada)


## 4. Sin cobertura: redacta la aclaración

Si ningún chunk supera el umbral, el agente llama al modelo para redactar un
mensaje. El prompt se **ancla al título y resumen** del documento: el investigador
no inventa contenido, solo explica qué SÍ trata el documento.


In [ ]:
generador_aclaracion = FakeGenerator(
    {"mensaje": "El documento trata sobre redes, ¿querés seguir con ese tema?"}
)
recuperador = FakeRecuperador([CHUNK_BAJO])

resultado = await investigar(
    "Kubernetes",
    recuperador,
    generador_aclaracion,
    TITULO,
    resumen_documento="Redes privadas y subredes.",
)

print("fuente_confirmada:", resultado.fuente_confirmada)
print("mensaje:", resultado.mensaje_aclaracion)
print("modelo pedido:", generador_aclaracion.output_model.__name__)
print("¿el prompt incluye el título?:", TITULO in generador_aclaracion.prompt)


## 5. Los errores del contrato se rechazan

El agente valida en los bordes: tema vacío, chunk inválido y salida inválida
tienen excepciones propias. Nada se ignora en silencio.


In [ ]:
from src.errores import ErrorLLM, ErrorSalidaInvalida

# Tema vacío: ni siquiera consulta el recuperador.
recuperador = FakeRecuperador([CHUNK_MATCH])
try:
    await investigar("   ", recuperador, FakeGenerator({}), TITULO)
except ErrorSalidaInvalida as exc:
    print("tema vacío ->", exc, "| consultó?:", recuperador.consulta is not None)

# Chunk con score fuera de rango.
try:
    await investigar(
        "VCN",
        FakeRecuperador([{"id": "c", "texto": "x", "score": 2.0}]),
        FakeGenerator({}),
        TITULO,
    )
except ErrorSalidaInvalida as exc:
    print("chunk inválido ->", exc)

# El modelo devuelve un mensaje vacío.
try:
    await investigar(
        "Kubernetes", FakeRecuperador([CHUNK_BAJO]),
        FakeGenerator({"mensaje": "   "}), TITULO,
    )
except ErrorSalidaInvalida as exc:
    print("mensaje vacío ->", exc)

# El generador falla (p. ej. proveedor caído).
try:
    await investigar(
        "Kubernetes", FakeRecuperador([CHUNK_BAJO]),
        FakeGenerator(RuntimeError("offline")), TITULO,
    )
except ErrorLLM as exc:
    print("fallo del LLM ->", exc)


## 6. Opcional — la aclaración con un LLM real

El mismo núcleo, sin cambiar nada, corre con el modelo real. Solo se ejecuta si
tienes `LLM_PROVIDER` configurado. Recuerda: en producción el `recuperador` real
viene de `rag/vectorstore.py` (Chroma), no de un doble.


In [ ]:
import os

if os.getenv("LLM_PROVIDER"):
    from src.agentes.generador_langchain import GeneradorLangchain
    from src.llm_provider import get_llm

    generador_real = GeneradorLangchain(get_llm())
    resultado_real = await investigar(
        "Kubernetes",
        FakeRecuperador([CHUNK_BAJO]),
        generador_real,
        TITULO,
        resumen_documento="Redes privadas y subredes.",
    )
    print(resultado_real.model_dump())
else:
    print(
        "Sin LLM_PROVIDER configurado: se omite la prueba real "
        "(no hace falta para el resto del notebook)."
    )


## Qué aprendimos

- La **cobertura** se decide de forma determinística por similitud (Chroma), no
  por el LLM. El modelo solo redacta cuando NO hay cobertura.
- El agente recibe **ambas dependencias inyectadas** (`recuperador` y
  `generador`): se prueba sin red ni cuota, y en producción se le pasa Chroma real.
- El mensaje de aclaración se **ancla al documento**; el investigador no inventa
  contenido educativo.

**Cómo se extiende:** cuando `rag/vectorstore.py` esté listo, se pasa su
recuperador real en `app.py`. El agente no se toca.
